In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f


Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/29 14:48:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../../data/25.06/"
path_to_intermediate_data_folder = "../../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

all_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence")


efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)


g_p_s = session.spark.read.parquet(path_to_intermediate_data_folder + "genes_therapeutic_areas")
g_p_s.count()


26/01/29 14:48:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

8285

In [4]:
l2g_full.show(1)


+--------------------+--------------------+---------------+------------------+----------+----------+---+-----------+-------------------+--------------+-------------------+----+------+--------------------+----------+--------------+----+
|        studyLocusId|             studyId|         geneId|             score|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|                maf|     variantId|            absBeta|year|is_nfe|          diseaseIds|nfe_common|non_nfe_common|rare|
+--------------------+--------------------+---------------+------------------+----------+----------+---+-----------+-------------------+--------------+-------------------+----+------+--------------------+----------+--------------+----+
|8bb394926da96ce7e...|FINNGEN_R12_AB1_V...|ENSG00000128052|0.4011244773864746|         0|         0|  0|          1|0.15027322404371585|4_55014612_A_G|0.28786060190050455|2024|     0|[MONDO_0024294, E...|         0|             1|   0|
+--------------------+--------------------+-------------

In [4]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pyspark.sql import DataFrame
from statsmodels.stats.contrast import Contrast


def l2g_full_to_indirect_assosiations_max(
    l2g_full_not_exploded: DataFrame,
    disease_index_orig: DataFrame,
    efo_to_remove: list[str] | None = None,
) -> DataFrame:
    disease_target_evidence = (
        l2g_full_not_exploded.withColumn("diseaseId", f.explode("diseaseIds"))
        .withColumnRenamed("geneId", "targetId")
        .drop("diseaseIds")
    )

    if efo_to_remove is not None:
        disease_target_evidence = disease_target_evidence.filter(~f.col("diseaseId").isin(efo_to_remove))

    disease_index = disease_index_orig.select(
        f.col("id").alias("diseaseId"),
        f.explode("ancestors").alias("ancestorDiseaseId"),
    )

    disease_index = disease_index.union(
        disease_index_orig.select(
            f.col("id").alias("diseaseId"),
            f.col("id").alias("ancestorDiseaseId"),
        )
    )
    return (
        disease_target_evidence.join(disease_index, on="diseaseId", how="inner")
        .groupBy("targetId", "ancestorDiseaseId")
        .agg(
            f.max("absBeta").alias("max_beta"),
            f.min("maf").alias("min_maf"),
            f.max("VEP").alias("max_vep"),
            f.max("score").alias("indirect_assoc_score"),
        )
        .select("targetId", "ancestorDiseaseId", "indirect_assoc_score", "max_beta", "min_maf", "max_vep")
        .withColumnRenamed("ancestorDiseaseId", "diseaseId")
    )


def drug_enrichemnt_from_evidence_dataframe(
    l2g_full_not_exploded: DataFrame,
    chembl_orig: DataFrame,
    gPS: DataFrame,
    indirect_assoc_score_thr: float = 0.5,
    efo_ancestors_to_remove: list[str] | None = None,
) -> pd.DataFrame:
    """Run chembl drug enrichment from scores.

    Args:
        evid (DataFrame): Evidence table
        sub_evid (DataFrame): Subset of evidence table
        disease_index_orig (DataFrame): The original disease index (not epxloded)
        chembl_orig (DataFrame): Chembl evidence
        indirect_assoc_score_thr (float): Minimum score to keep in indirect associations
        efo_ancestors_to_remove (list[str] | None): List of EFO IDs to remove
    Returns:
        pd.DataFrame: Drug enrichment table.

    """
    if efo_ancestors_to_remove is not None:
        efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
            disease_index_orig=disease_index_orig,
            efo_ids=efo_ancestors_to_remove,
        )
    else:
        efo_to_remove = None

    chembl = chemblDrugEnrichment.process_chembl_evidence(chembl_orig, efo_to_remove)

    evid_indirect = l2g_full_to_indirect_assosiations_max(
        l2g_full_not_exploded=l2g_full_not_exploded,
        disease_index_orig=disease_index_orig,
        efo_to_remove=efo_to_remove,
    ).cache()

    joined_data = evid_indirect.join(chembl, ["targetId", "diseaseId"], "right")

    joined_data = joined_data.join(
        gPS.select("targetId", "uniqueDiseases", "uniqueTherapeuticAreas"),
        on="targetId",
        how="left",
    )

    joined_data = joined_data.toPandas()
    joined_data = joined_data.fillna(
        {
            "indirect_assoc_score": 0,
            "uniqueDiseases": 0,
            "uniqueTherapeuticAreas": 0,
            "max_beta": 0,
            "min_maf": 0,
            "max_vep": 0,
        }
    )

    joined_data["outcome"] = (joined_data["maxClinicalPhase"] >= 4).astype(int)
    joined_data["geneticSupport"] = (joined_data["indirect_assoc_score"] >= indirect_assoc_score_thr).astype(int)

    return joined_data


In [5]:
full_l2g_evidnce = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full.drop("diseaseIds"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)
full_l2g_evidnce.count()


77071

In [6]:
df_for_regression = drug_enrichemnt_from_evidence_dataframe(
    l2g_full_not_exploded=l2g_full,
    gPS=g_p_s.withColumnRenamed("geneId", "targetId"),
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.1,
    efo_ancestors_to_remove=["MONDO_0045024"],
)


In [7]:
df_for_regression


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport
0,ENSG00000004779,MONDO_0020121,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0
1,ENSG00000011677,EFO_0005411,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0
2,ENSG00000018625,EFO_0000544,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
3,ENSG00000022355,EFO_0007634,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0
4,ENSG00000053918,EFO_0000546,0.0,0.0,0.0,0.0,2.0,18.0,11.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
37372,ENSG00000273079,EFO_0003108,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0
37373,ENSG00000274286,EFO_0005207,0.0,0.0,0.0,0.0,3.0,1.0,1.0,0,0
37374,ENSG00000274286,MONDO_0001330,0.0,0.0,0.0,0.0,3.0,1.0,1.0,0,0
37375,ENSG00000274286,Orphanet_1764,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0


In [ ]:
df_for_regression.to_csv(path_to_intermediate_data_folder + "df_for_enrichment_regression.csv", index=False)


# Load the data and check non linearity of MAF and Beta


In [ ]:
df_for_regression = pd.read_csv(path_to_intermediate_data_folder + "df_for_enrichment_regression.csv")


In [11]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+uniqueDiseases", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37374
Method:                           MLE   Df Model:                            2
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.007826
Time:                        15:19:19   Log-Likelihood:                -13762.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 7.176e-48
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0086      0.018   -114.439      0.000      -2.043      -1.974
geneticSupport     1.2988      0.084     15.381      0.000       1.133       1.464
uniqueDiseases    -0.0012      0.002

In [12]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+uniqueDiseases+I(uniqueDiseases ** 2)", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37373
Method:                           MLE   Df Model:                            3
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.008100
Time:                        15:19:42   Log-Likelihood:                -13758.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 1.943e-48
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -2.0268      0.019   -107.577      0.000      -2.064      -1.990
geneticSupport             1.2713      0.085     14.943      0.000       1.105       1.438
uniq

In [13]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+max_beta", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37374
Method:                           MLE   Df Model:                            2
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.008032
Time:                        15:19:51   Log-Likelihood:                -13759.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 4.116e-49
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0118      0.016   -124.210      0.000      -2.043      -1.980
geneticSupport     1.1497      0.099     11.667      0.000       0.957       1.343
max_beta           0.5341      0.219

In [14]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+max_beta+I(max_beta**2)", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37373
Method:                           MLE   Df Model:                            3
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.008042
Time:                        15:20:09   Log-Likelihood:                -13759.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 4.286e-48
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -2.0118      0.016   -124.210      0.000      -2.043      -1.980
geneticSupport       1.1171      0.116      9.592      0.000       0.889       1.345
max_beta             0.7730 

In [15]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+max_vep", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37374
Method:                           MLE   Df Model:                            2
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.008295
Time:                        15:20:15   Log-Likelihood:                -13756.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 1.068e-50
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0118      0.016   -124.210      0.000      -2.043      -1.980
geneticSupport     1.1290      0.093     12.190      0.000       0.947       1.310
max_vep            0.6708      0.183

In [16]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+min_maf", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37374
Method:                           MLE   Df Model:                            2
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.008293
Time:                        15:20:21   Log-Likelihood:                -13756.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 1.107e-50
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0118      0.016   -124.210      0.000      -2.043      -1.980
geneticSupport     1.6510      0.127     13.025      0.000       1.403       1.899
min_maf           -1.9530      0.546

In [17]:
df = df_for_regression.copy()
model = smf.logit("outcome ~ geneticSupport+min_maf+I(min_maf**2)", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37373
Method:                           MLE   Df Model:                            3
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.008358
Time:                        15:20:33   Log-Likelihood:                -13755.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 5.500e-50
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -2.0118      0.016   -124.210      0.000      -2.043      -1.980
geneticSupport      1.7853      0.161     11.093      0.000       1.470       2.101
min_maf            -4.4874      

In [18]:
model2 = smf.logit("outcome ~ geneticSupport+max_beta+I(max_beta ** 2)", data=df).fit(disp=False)
model1 = smf.logit("outcome ~ geneticSupport+max_beta", data=df).fit(disp=False)
model0 = smf.logit("outcome ~ geneticSupport", data=df).fit(disp=False)
from scipy.stats import chi2


def lr_test(model_full, model_restricted):
    """Likelihood-ratio test for nested models.
    Returns (LR_stat, pvalue, df_diff).
    """
    llf_full = float(model_full.llf)
    llf_red = float(model_restricted.llf)
    k_full = len(model_full.params)
    k_red = len(model_restricted.params)
    df_diff = k_full - k_red
    if df_diff <= 0:
        raise ValueError("models are not nested in the expected direction (full must have more params)")
    LR = 2.0 * (llf_full - llf_red)
    pval = chi2.sf(LR, df_diff)
    return LR, pval, df_diff


# compare models
LR20, p20, df20 = lr_test(model2, model0)  # model2 vs model0
LR10, p10, df10 = lr_test(model1, model0)  # model1 vs model0
LR21, p21, df21 = lr_test(model2, model1)  # model2 vs model1

print(f"model2 vs model0: LR={LR20:.3f}, df={df20}, p={p20:.3g}")
print(f"model1 vs model0: LR={LR10:.3f}, df={df10}, p={p10:.3g}")
print(f"model2 vs model1: LR={LR21:.3f}, df={df21}, p={p21:.3g}")
# ...existing code...


model2 vs model0: LR=6.215, df=2, p=0.0447
model1 vs model0: LR=5.936, df=1, p=0.0148
model2 vs model1: LR=0.279, df=1, p=0.597


In [19]:
model2 = smf.logit("outcome ~ geneticSupport+min_maf+I(min_maf ** 2)", data=df).fit(disp=False)
model1 = smf.logit("outcome ~ geneticSupport+min_maf", data=df).fit(disp=False)
model0 = smf.logit("outcome ~ geneticSupport", data=df).fit(disp=False)
from scipy.stats import chi2


def lr_test(model_full, model_restricted):
    """Likelihood-ratio test for nested models.
    Returns (LR_stat, pvalue, df_diff).
    """
    llf_full = float(model_full.llf)
    llf_red = float(model_restricted.llf)
    k_full = len(model_full.params)
    k_red = len(model_restricted.params)
    df_diff = k_full - k_red
    if df_diff <= 0:
        raise ValueError("models are not nested in the expected direction (full must have more params)")
    LR = 2.0 * (llf_full - llf_red)
    pval = chi2.sf(LR, df_diff)
    return LR, pval, df_diff


# compare models
LR20, p20, df20 = lr_test(model2, model0)  # model2 vs model0
LR10, p10, df10 = lr_test(model1, model0)  # model1 vs model0
LR21, p21, df21 = lr_test(model2, model1)  # model2 vs model1

print(f"model2 vs model0: LR={LR20:.3f}, df={df20}, p={p20:.3g}")
print(f"model1 vs model0: LR={LR10:.3f}, df={df10}, p={p10:.3g}")
print(f"model2 vs model1: LR={LR21:.3f}, df={df21}, p={p21:.3g}")
# ...existing code...


model2 vs model0: LR=14.965, df=2, p=0.000563
model1 vs model0: LR=13.168, df=1, p=0.000285
model2 vs model1: LR=1.797, df=1, p=0.18


# Clement comments - correlashion


In [ ]:
df_for_regression = pd.read_csv(path_to_intermediate_data_folder + "df_for_enrichment_regression.csv")


In [21]:
df = df_for_regression[df_for_regression["outcome"] == 1].copy()


In [22]:
df


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport
9,ENSG00000073756,EFO_0000729,0.0,0.0,0.0,0.0,4.0,3.0,1.0,1,0
13,ENSG00000081248,HP_0000103,0.0,0.0,0.0,0.0,4.0,4.0,4.0,1,0
14,ENSG00000082175,HP_0002039,0.0,0.0,0.0,0.0,4.0,6.0,4.0,1,0
32,ENSG00000109158,EFO_0000474,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1,0
51,ENSG00000124491,Orphanet_183654,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
37358,ENSG00000196876,MONDO_0005277,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1,0
37359,ENSG00000197635,MONDO_0021187,0.0,0.0,0.0,0.0,4.0,7.0,4.0,1,0
37360,ENSG00000198216,EFO_0000319,0.0,0.0,0.0,0.0,4.0,1.0,1.0,1,0
37367,ENSG00000232810,EFO_0000540,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1,0


In [25]:
# Basic aggregation
df_agg = (
    df.groupby("targetId")
    .agg(
        num_rows=("targetId", "count"),
        max_uniqueDiseases=("uniqueDiseases", "max"),
        max_uniqueTherapeuticAreas=("uniqueTherapeuticAreas", "max"),
    )
    .reset_index()
)


In [26]:
df_agg


,targetId,num_rows,max_uniqueDiseases,max_uniqueTherapeuticAreas
0,ENSG00000001626,4,3.0,3.0
1,ENSG00000003436,1,2.0,2.0
2,ENSG00000004779,2,0.0,0.0
3,ENSG00000004948,5,1.0,1.0
4,ENSG00000005844,4,5.0,3.0
...,...,...,...,...
749,ENSG00000273079,16,1.0,1.0
750,ENSG00000274286,48,1.0,1.0
751,ENSG00000277893,2,2.0,2.0
752,ENSG00000278195,2,0.0,0.0


In [ ]:
from scipy.stats import pearsonr

df_agg_2 = df_agg[df_agg["max_uniqueDiseases"] > 1]
# Calculate correlations and p-values
vars_to_correlate = ["num_rows", "max_uniqueDiseases", "max_uniqueTherapeuticAreas"]

print("Correlation Matrix and P-values:")
print("=" * 70)
for i in range(len(vars_to_correlate)):
    for j in range(i + 1, len(vars_to_correlate)):
        var1 = vars_to_correlate[i]
        var2 = vars_to_correlate[j]
        corr, pval = pearsonr(df_agg_2[var1], df_agg_2[var2])
        print(f"{var1} vs {var2}:")
        print(f"  Correlation: {corr:.4f}, p-value: {pval:.4e}\n")


Correlation Matrix and P-values:
num_rows vs max_uniqueDiseases:
  Correlation: -0.0028, p-value: 9.6264e-01

num_rows vs max_uniqueTherapeuticAreas:
  Correlation: -0.0178, p-value: 7.6597e-01

max_uniqueDiseases vs max_uniqueTherapeuticAreas:
  Correlation: 0.8380, p-value: 2.2673e-75



In [ ]:
len(df_agg_2)


281

In [32]:
good_targets = df_for_regression[df_for_regression["outcome"] == 1]["targetId"].unique()


In [34]:
df_with_good_target = df_for_regression.copy()


In [35]:
df_with_good_target["good_target"] = 0
df_with_good_target.loc[df_with_good_target["targetId"].isin(good_targets), "good_target"] = 1


# Repurposing


In [39]:
df_for_regression = pd.read_csv(path_to_intermediate_data_folder + "df_for_enrichment_regression.csv")


In [40]:
good_targets = df_for_regression[df_for_regression["outcome"] == 1]["targetId"].unique()


In [41]:
df_with_good_target = df_for_regression.copy()


In [42]:
df_with_good_target["good_target"] = 0
df_with_good_target.loc[df_with_good_target["targetId"].isin(good_targets), "good_target"] = 1


In [ ]:
# Calculate number of unique diseaseId with outcome==1 for each targetId
successful_indications = (
    df_with_good_target[df_with_good_target["outcome"] == 1]
    .groupby("targetId")["diseaseId"]
    .nunique()
    .reset_index()
    .rename(columns={"diseaseId": "number_of_successful_indications"})
)

# Merge back to df_with_good_target
df_with_good_target = df_with_good_target.merge(successful_indications, on="targetId", how="left")

# Fill NaN values (targets with no successful indications) with 0
df_with_good_target["number_of_successful_indications"] = (
    df_with_good_target["number_of_successful_indications"].fillna(0).astype(int)
)


In [ ]:
df_with_good_target


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport,good_target,number_of_successful_indications
0,ENSG00000004779,MONDO_0020121,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0,1,2
1,ENSG00000011677,EFO_0005411,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0,1,23
2,ENSG00000018625,EFO_0000544,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0,1,4
3,ENSG00000022355,EFO_0007634,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0,1,23
4,ENSG00000053918,EFO_0000546,0.0,0.0,0.0,0.0,2.0,18.0,11.0,0,0,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37372,ENSG00000273079,EFO_0003108,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0,1,16
37373,ENSG00000274286,EFO_0005207,0.0,0.0,0.0,0.0,3.0,1.0,1.0,0,0,1,48
37374,ENSG00000274286,MONDO_0001330,0.0,0.0,0.0,0.0,3.0,1.0,1.0,0,0,1,48
37375,ENSG00000274286,Orphanet_1764,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0,1,48


In [ ]:
from scipy.stats import pearsonr

df_agg_2 = df_with_good_target[df_with_good_target["good_target"] == 1]
df_agg_2_distinct = df_agg_2[
    ["targetId", "number_of_successful_indications", "uniqueDiseases", "uniqueTherapeuticAreas"]
].drop_duplicates()
df_agg_2_distinct = df_agg_2_distinct[df_agg_2_distinct["uniqueDiseases"] > 0]
# Calculate correlations and p-values
vars_to_correlate = ["number_of_successful_indications", "uniqueDiseases", "uniqueTherapeuticAreas"]

print("Correlation Matrix and P-values:")
print("=" * 70)
for i in range(len(vars_to_correlate)):
    for j in range(i + 1, len(vars_to_correlate)):
        var1 = vars_to_correlate[i]
        var2 = vars_to_correlate[j]
        corr, pval = pearsonr(df_agg_2_distinct[var1], df_agg_2_distinct[var2])
        print(f"{var1} vs {var2}:")
        print(f"  Correlation: {corr:.4f}, p-value: {pval:.4e}\n")


Correlation Matrix and P-values:
number_of_successful_indications vs uniqueDiseases:
  Correlation: -0.0035, p-value: 9.4411e-01

number_of_successful_indications vs uniqueTherapeuticAreas:
  Correlation: -0.0148, p-value: 7.6729e-01

uniqueDiseases vs uniqueTherapeuticAreas:
  Correlation: 0.8665, p-value: 9.0593e-123



In [ ]:
df_with_good_target_multi_indic = df_with_good_target[df_with_good_target["number_of_successful_indications"] > 1]


In [ ]:
df_with_good_target_multi_indic


,targetId,diseaseId,indirect_assoc_score,max_beta,min_maf,max_vep,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,outcome,geneticSupport,good_target,number_of_successful_indications
0,ENSG00000004779,MONDO_0020121,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0,1,2
1,ENSG00000011677,EFO_0005411,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0,0,1,23
2,ENSG00000018625,EFO_0000544,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0,1,4
3,ENSG00000022355,EFO_0007634,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0,1,23
4,ENSG00000053918,EFO_0000546,0.0,0.0,0.0,0.0,2.0,18.0,11.0,0,0,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37372,ENSG00000273079,EFO_0003108,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0,1,16
37373,ENSG00000274286,EFO_0005207,0.0,0.0,0.0,0.0,3.0,1.0,1.0,0,0,1,48
37374,ENSG00000274286,MONDO_0001330,0.0,0.0,0.0,0.0,3.0,1.0,1.0,0,0,1,48
37375,ENSG00000274286,Orphanet_1764,0.0,0.0,0.0,0.0,2.0,1.0,1.0,0,0,1,48


In [56]:
df = df_with_good_target_multi_indic.copy()
model = smf.logit("outcome ~ geneticSupport", data=df).fit(disp=False)
print(model.summary())


                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                32381
Model:                          Logit   Df Residuals:                    32379
Method:                           MLE   Df Model:                            1
Date:                Thu, 29 Jan 2026   Pseudo R-squ.:                0.009930
Time:                        15:44:52   Log-Likelihood:                -12743.
converged:                       True   LL-Null:                       -12871.
Covariance Type:            nonrobust   LLR p-value:                 1.561e-57
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -1.8918      0.017   -113.824      0.000      -1.924      -1.859
geneticSupport     1.4821      0.086     17.280      0.000       1.314       1.650


In [ ]:
np.exp(1.4821)


np.float64(4.402180560077985)